# 面试问题：SFT 的 Chat Template、Loss Mask 与 Packing 怎样从零实现？

可直接复述的回答：训练模板必须与服务模板完全一致，否则模型学习和推理看到的角色边界不同。每段消息要显式加入角色 token 和结束 token。监督标签通常只覆盖 assistant 内容，system、user、padding 和跨样本边界设为 ignore index。Causal loss 使用 logits 的前 T-1 位预测 labels 的后 T-1 位。Packing 提高利用率，但注意力必须按样本分块，不能让一个对话看到下一个对话。截断应优先保留监督答案并记录丢弃率。发布时要绑定 tokenizer、模板和 mask 版本。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：客服对话与输入预览

五段脱敏客服对话保留 system、user 和 assistant 文本。词语用空格分开便于展示 token 对齐；这是教学语料，不代表真实 SFT 规模和语言分布。


In [1]:
import torch  # 使用 PyTorch 基础张量实现标签和训练。
from torch import nn  # 使用基础模块构造微型语言头。
import torch.nn.functional as F  # 使用交叉熵计算 masked causal loss。
torch.manual_seed(20260729)  # 固定参数初始化保证输出确定。
dialogues10 = [  # 构造五段具有真实角色语义的客服对话。
    {"system": "你 是 客服", "user": "订单 延迟 怎么办", "assistant": "我 来 查询 物流 状态"},  # 物流查询回答。
    {"system": "你 是 客服", "user": "如何 申请 退款", "assistant": "请 提供 订单号 并 确认 原因"},  # 退款流程回答。
    {"system": "你 是 客服", "user": "会员 积分 过期 吗", "assistant": "积分 有效期 以 最新 规则 为准"},  # 规则引用回答。
    {"system": "你 是 客服", "user": "修改 收货 地址", "assistant": "发货 前 可以 提交 地址 变更"},  # 地址修改回答。
    {"system": "你 是 客服", "user": "商品 破损", "assistant": "请 上传 图片 我们 会 优先 处理"},  # 售后证据回答。
]  # 完成五条对话记录。
print("教学实验输入：system | user | assistant")  # 输出对话预览表头。
for dialogue10 in dialogues10:  # 逐条展示原始角色内容。
    print(dialogue10)  # 输出一段训练对话。


教学实验输入：system | user | assistant
{'system': '你 是 客服', 'user': '订单 延迟 怎么办', 'assistant': '我 来 查询 物流 状态'}
{'system': '你 是 客服', 'user': '如何 申请 退款', 'assistant': '请 提供 订单号 并 确认 原因'}
{'system': '你 是 客服', 'user': '会员 积分 过期 吗', 'assistant': '积分 有效期 以 最新 规则 为准'}
{'system': '你 是 客服', 'user': '修改 收货 地址', 'assistant': '发货 前 可以 提交 地址 变更'}
{'system': '你 是 客服', 'user': '商品 破损', 'assistant': '请 上传 图片 我们 会 优先 处理'}


## 2. Baseline（基线）：所有 token 都计算 loss

朴素模板把角色、用户问题和答案全部设为监督目标。这样模型被要求复述用户输入和 system 文本，监督预算被非答案 token 稀释。


In [2]:
special10 = ["<pad>", "<system>", "<user>", "<assistant>", "<eos>"]  # 定义模板需要的特殊 token。
words10 = sorted({word10 for dialogue10 in dialogues10 for field10 in ["system", "user", "assistant"] for word10 in dialogue10[field10].split()})  # 收集教学语料词表。
vocabulary10 = {token10: index10 for index10, token10 in enumerate(special10 + words10)}  # 建立确定性 token 到 ID 映射。
inverse10 = {index10: token10 for token10, index10 in vocabulary10.items()}  # 建立 ID 到 token 的反向映射。
def render10(dialogue10):  # 按固定 Chat Template 渲染一段对话。
    tokens10 = ["<system>"] + dialogue10["system"].split() + ["<eos>"]  # 渲染 system 片段。
    tokens10 += ["<user>"] + dialogue10["user"].split() + ["<eos>"]  # 追加 user 片段。
    tokens10 += ["<assistant>"] + dialogue10["assistant"].split() + ["<eos>"]  # 追加 assistant 片段。
    ids10 = torch.tensor([vocabulary10[token10] for token10 in tokens10], dtype=torch.long)  # 将可读 token 转成 ID 张量。
    return tokens10, ids10  # 返回模板 token 和 ID。
rendered10 = [render10(dialogue10) for dialogue10 in dialogues10]  # 渲染全部对话。
baseline_labels10 = [ids10.clone() for _, ids10 in rendered10]  # 错误地监督所有角色和内容 token。
print("基线首条模板", rendered10[0][0])  # 展示角色 token 的真实排列。
print("基线监督token数", int(sum(labels10.numel() for labels10 in baseline_labels10)), "总token数", int(sum(ids10.numel() for _, ids10 in rendered10)))  # 展示所有 token 都进入 loss。


基线首条模板 ['<system>', '你', '是', '客服', '<eos>', '<user>', '订单', '延迟', '怎么办', '<eos>', '<assistant>', '我', '来', '查询', '物流', '状态', '<eos>']
基线监督token数 90 总token数 90


## 3. 核心实现：assistant-only Loss Mask 与 causal shift

为每个 token 构造同长布尔 mask，只让 assistant 正文及其结束 token 保留标签，其余位置设为 -100。输出首条 token-label 对照，直观看到角色和用户位置不参与 loss。


In [3]:
masked_examples10 = []  # 收集 ID、标签和可读 token 对照。
for dialogue10, (tokens10, ids10) in zip(dialogues10, rendered10):  # 逐对话构造监督掩码。
    assistant_start10 = tokens10.index("<assistant>")  # 定位 assistant 角色起点。
    labels10 = torch.full_like(ids10, -100)  # 默认忽略所有模板和输入 token。
    labels10[assistant_start10 + 1:] = ids10[assistant_start10 + 1:]  # 只监督 assistant 正文和结束 token。
    masked_examples10.append((ids10, labels10, tokens10))  # 保存当前对话的训练张量。
first_ids10, first_labels10, first_tokens10 = masked_examples10[0]  # 读取首条对话用于可视化。
alignment10 = []  # 收集 token 与训练标签对照。
for position10, (token10, label10) in enumerate(zip(first_tokens10, first_labels10.tolist())):  # 逐位置解释 loss mask。
    target10 = "IGNORE" if label10 == -100 else inverse10[label10]  # 把标签 ID 转回可读 token。
    alignment10.append((position10, token10, target10))  # 保存位置、输入和监督目标。
print("首条对话 mask：position | input_token | label")  # 输出标签对齐表头。
for row10 in alignment10:  # 逐位置展示是否参与 loss。
    print(row10)  # 输出一行 token-label 对照。


首条对话 mask：position | input_token | label
(0, '<system>', 'IGNORE')
(1, '你', 'IGNORE')
(2, '是', 'IGNORE')
(3, '客服', 'IGNORE')
(4, '<eos>', 'IGNORE')
(5, '<user>', 'IGNORE')
(6, '订单', 'IGNORE')
(7, '延迟', 'IGNORE')
(8, '怎么办', 'IGNORE')
(9, '<eos>', 'IGNORE')
(10, '<assistant>', 'IGNORE')
(11, '我', '我')
(12, '来', '来')
(13, '查询', '查询')
(14, '物流', '物流')
(15, '状态', '状态')
(16, '<eos>', '<eos>')


## 4. 结果表、Packing 与结果解读

将前三条对话拼成一个序列，并用 segment ID 构造块对角 causal mask。微型 Embedding+Linear 只用于验证 masked causal loss 能下降，不冒充 Transformer。


In [4]:
packed_ids10 = torch.cat([item10[0] for item10 in masked_examples10[:3]])  # 拼接前三条对话 ID。
packed_labels10 = torch.cat([item10[1] for item10 in masked_examples10[:3]])  # 拼接对应 assistant-only 标签。
segment_ids10 = torch.cat([torch.full((item10[0].numel(),), index10, dtype=torch.long) for index10, item10 in enumerate(masked_examples10[:3])])  # 为每条对话标记独立 segment。
positions10 = torch.arange(packed_ids10.numel())  # 创建拼接序列位置索引。
block_causal10 = (segment_ids10[:, None] == segment_ids10[None, :]) & (positions10[:, None] >= positions10[None, :])  # 同时限制同样本和因果方向。
class TinySFT10(nn.Module):  # 定义仅用于验证 loss mask 的微型语言头。
    def __init__(self, vocab_size10, hidden10=32):  # 初始化嵌入和输出投影。
        super().__init__()  # 注册 PyTorch 模块状态。
        self.embedding = nn.Embedding(vocab_size10, hidden10)  # 将 token ID 映射到隐藏向量。
        self.projection = nn.Linear(hidden10, vocab_size10)  # 把隐藏向量映射到下一 token logits。
    def forward(self, ids10):  # 定义前向计算。
        return self.projection(self.embedding(ids10))  # 返回每个位置的词表 logits。
model10 = TinySFT10(len(vocabulary10))  # 创建确定性微型训练模型。
optimizer10 = torch.optim.Adam(model10.parameters(), lr=0.08)  # 使用基础优化器训练教学模型。
loss_trace10 = []  # 保存 masked causal loss 变化。
for _ in range(35):  # 运行少量确定性训练步骤。
    optimizer10.zero_grad()  # 清空上一步梯度。
    logits10 = model10(packed_ids10)  # 计算拼接序列 logits。
    loss10 = F.cross_entropy(logits10[:-1], packed_labels10[1:], ignore_index=-100)  # 使用 causal shift 和 ignore index 计算 loss。
    loss10.backward()  # 反向传播到微型模型参数。
    optimizer10.step()  # 更新嵌入和投影参数。
    loss_trace10.append(float(loss10.detach()))  # 保存当前训练损失。
print("Packing形状", tuple(packed_ids10.shape), "block mask形状", tuple(block_causal10.shape), "跨样本可见边", int(((segment_ids10[:, None] != segment_ids10[None, :]) & block_causal10).sum()))  # 展示拼接与隔离结果。
print("masked causal loss：start -> end", round(loss_trace10[0], 4), "->", round(loss_trace10[-1], 4))  # 展示核心训练过程。
print("结果解读：packing提升token利用率，segment mask防止跨对话注意力，-100只监督答案")  # 解释三个机制的分工。


Packing形状 (54,) block mask形状 (54, 54) 跨样本可见边 0
masked causal loss：start -> end 4.3341 -> 0.1648
结果解读：packing提升token利用率，segment mask防止跨对话注意力，-100只监督答案


## 5. 失败案例与修正：跨样本边界被当成下一 token

直接对整条 packed IDs 做全局 shift，会让上一段 `<eos>` 预测下一段 `<system>`。修正是在 segment 边界把目标设为 -100，并在注意力中阻断跨段连接。


In [5]:
boundary10 = segment_ids10[:-1] != segment_ids10[1:]  # 找出相邻位置属于不同对话的边界。
naive_next10 = packed_ids10[1:].clone()  # 构造错误的全局下一 token 标签。
fixed_next10 = packed_labels10[1:].clone()  # 从 assistant-only 标签构造修正目标。
fixed_next10[boundary10] = -100  # 显式忽略跨样本下一 token。
boundary_rows10 = []  # 收集每个 packing 边界的错误和修正标签。
for position10 in torch.where(boundary10)[0].tolist():  # 遍历所有跨样本边界。
    boundary_rows10.append((position10, inverse10[int(packed_ids10[position10])], inverse10[int(naive_next10[position10])], int(fixed_next10[position10])))  # 保存前 token、错误目标和修正目标。
print("失败行为：boundary | previous | naive_next | fixed_label")  # 输出边界错误表头。
for row10 in boundary_rows10:  # 展示每个错误跨样本目标。
    print(row10)  # 输出一条边界修正记录。


失败行为：boundary | previous | naive_next | fixed_label
(16, '<eos>', '<system>', -100)
(34, '<eos>', '<system>', -100)


## 6. 生产边界与训练制品

真实 SFT 需要 tokenizer 的 offset mapping、长对话截断策略、padding、分布式 packing 和模板回归测试。这里只训练微型语言头，不证明模型泛化；模板、词表和 mask 规则必须随 checkpoint 发布。


In [6]:
sft_contract10 = {"template": "support-chat-v4", "tokenizer_vocab": len(vocabulary10), "loss_mask": "assistant_content_and_eos", "packing": "block_causal", "ignore_index": -100}  # 定义训练与服务必须共享的模板合同。
print("SFT 训练制品", sft_contract10)  # 展示模板、mask 和 packing 版本字段。
print("生产替换点：真实tokenizer offsets、截断审计、FlashAttention block mask、分布式packing和服务模板一致性")  # 说明教学实现的边界。


SFT 训练制品 {'template': 'support-chat-v4', 'tokenizer_vocab': 50, 'loss_mask': 'assistant_content_and_eos', 'packing': 'block_causal', 'ignore_index': -100}
生产替换点：真实tokenizer offsets、截断审计、FlashAttention block mask、分布式packing和服务模板一致性


## 7. 最小回归测试

断言保护模板规模、loss mask、packing 隔离和训练可运行性。


In [7]:
assert len(dialogues10) >= 5  # 保证案例仍包含足够多的真实语义对话。
assert int((first_labels10 != -100).sum()) < first_labels10.numel()  # 保证 system 和 user 不参与监督。
assert int(((segment_ids10[:, None] != segment_ids10[None, :]) & block_causal10).sum()) == 0  # 保证注意力不跨样本泄漏。
assert all(int(value10) == -100 for value10 in fixed_next10[boundary10])  # 保证 packing 边界不产生错误目标。
assert loss_trace10[-1] < loss_trace10[0]  # 保证微型 masked causal 训练能够下降。
print("最小回归测试通过：模板、loss mask、packing边界和训练链路稳定")  # 显示 SFT 关键性质已验证。


最小回归测试通过：模板、loss mask、packing边界和训练链路稳定
